# Speculative Decoding

This notebook runs two speculative decoding experiments:
- **Pair 2:** TinyLlama-1.1B (draft) → Llama2-7B (target) — same Llama tokenizer ✅
- **Pair 4:** Mamba-130m (draft) → Llama2-7B (target) — cross-architecture proxy ⚠️

> **GPU Required:** Runtime → Change runtime type → T4 GPU (free) or A100 (Colab Pro)

## Step 1 — Verify GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU found — go to Runtime > Change runtime type > T4 GPU")

import torch
print(f"\nPyTorch   : {torch.__version__}")
print(f"CUDA      : {torch.version.cuda}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU       : {p.name}")
    print(f"VRAM      : {p.total_memory/1e9:.1f} GB")
    vram = p.total_memory / 1e9
    if vram >= 15:
        print("Status    : ✓ T4/A100 detected — sufficient for int8 Llama2-7B")
    else:
        print(f"WARNING   : Only {vram:.1f} GB — may be tight. Use int8.")
else:
    print("ERROR: No CUDA GPU. Please enable GPU runtime.")

## Step 2 — Install Dependencies

In [ ]:
%%capture
!pip install transformers==4.44.0 accelerate==0.27.0 bitsandbytes sentencepiece
!pip install matplotlib numpy
print("Installation complete")

In [ ]:
# Verify installs
import transformers, accelerate, bitsandbytes
print(f"transformers : {transformers.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"bitsandbytes : {bitsandbytes.__version__}")
print("All packages ready ✓")

## Step 3 — HuggingFace Login (Required for Llama2)

Llama2 is a gated model. You need to:
1. Go to https://huggingface.co/meta-llama/Llama-2-7b-hf
2. Accept Meta's license agreement
3. Get your token from https://huggingface.co/settings/tokens
4. Paste it below when prompted

In [ ]:
from huggingface_hub import login
login()
# Paste your HuggingFace token when prompted
# Token needs read access to gated models

In [ ]:
# Verify Llama2 access
from huggingface_hub import model_info
try:
    info = model_info("meta-llama/Llama-2-7b-hf")
    print(f"✓ Access confirmed: {info.modelId}")
except Exception as e:
    print(f"✗ Access denied: {e}")
    print("  Make sure you accepted the Llama2 license at huggingface.co/meta-llama/Llama-2-7b-hf")

## Step 4 — Core Speculative Decoding Functions

In [ ]:
import torch
import torch.nn.functional as F
import time
import gc
import json
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# ── VRAM helpers ──────────────────────────────────────────────
def vram_status(label=""):
    if not torch.cuda.is_available():
        return
    used  = torch.cuda.memory_allocated(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    free  = total - torch.cuda.memory_reserved(0) / 1e9
    tag   = f"[{label}] " if label else ""
    print(f"  {tag}VRAM: {used:.2f} GB used / {total:.1f} GB total / {free:.2f} GB free")

def free_vram(*models):
    for m in models:
        if m is not None:
            del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    time.sleep(1.0)
    vram_status("after cleanup")

# ── Model loading ─────────────────────────────────────────────
def load_model_int8(model_id, model_type="transformer"):
    """Load any model with int8 quantization (fits on T4 16GB)."""
    print(f"  Loading [{model_type}] {model_id}")

    tok = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    if model_type == "mamba":
        # Mamba doesn't support int8 — use float16 (only 0.26 GB anyway)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16, device_map=DEVICE
        )
    else:
        # int8 for Transformer models
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            load_in_8bit=True,
            device_map="auto",
        )

    model.eval()
    return model, tok

def load_model_float16(model_id, model_type="transformer"):
    """Load model in float16 — use only if you have 20+ GB VRAM."""
    tok   = AutoTokenizer.from_pretrained(model_id)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.float16, device_map="auto"
    )
    model.eval()
    return model, tok

# ── Measure beta ──────────────────────────────────────────────
def measure_beta(draft_model, target_model, tok, prompt, n_runs=8):
    inp = tok(prompt, return_tensors="pt").to(DEVICE)
    times_d, times_t = [], []
    for _ in range(n_runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            draft_model(inp.input_ids)
        torch.cuda.synchronize()
        times_d.append(time.perf_counter() - t0)

        torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.no_grad():
            target_model(inp.input_ids)
        torch.cuda.synchronize()
        times_t.append(time.perf_counter() - t0)

    # Drop first 2 (warmup)
    t_d = sum(times_d[2:]) / max(len(times_d[2:]), 1)
    t_t = sum(times_t[2:]) / max(len(times_t[2:]), 1)
    beta = t_d / t_t
    print(f"  Draft  : {t_d*1000:.1f} ms/pass")
    print(f"  Target : {t_t*1000:.1f} ms/pass")
    print(f"  β = {beta:.4f}  (draft costs {beta:.1%} of target)")
    return beta

# ── Draft functions ───────────────────────────────────────────
def draft_transformer(model, input_ids, k):
    draft_ids, draft_probs = [], []
    current = input_ids.clone()
    with torch.no_grad():
        for _ in range(k):
            out   = model(current)
            probs = F.softmax(out.logits[:, -1, :], dim=-1)
            token = torch.argmax(probs, dim=-1)
            draft_ids.append(token.item())
            draft_probs.append(max(probs[0, token.item()].item(), 1e-9))
            current = torch.cat([current, token.unsqueeze(0)], dim=1)
    return draft_ids, draft_probs

def draft_mamba(model, input_ids, k, target_vocab_size):
    """Mamba SSM draft — O(1) per step via recurrent state."""
    draft_ids, draft_probs = [], []
    device  = input_ids.device
    seq_len = input_ids.shape[1]

    def align(logits):
        vs = logits.shape[-1]
        if vs < target_vocab_size:
            pad = torch.full((logits.shape[0], target_vocab_size - vs),
                             float('-inf'), device=device, dtype=logits.dtype)
            return torch.cat([logits, pad], dim=-1)
        return logits[:, :target_vocab_size]

    with torch.no_grad():
        # Prefill: process full context
        out   = model(input_ids, use_cache=True)
        state = out.cache_params
        probs = F.softmax(align(out.logits[:, -1, :]), dim=-1)
        token = torch.argmax(probs, dim=-1)
        draft_ids.append(token.item())
        draft_probs.append(max(probs[0, token.item()].item(), 1e-9))

        # Recurrent decode — O(1) per step
        for step in range(k - 1):
            cache_pos = torch.tensor([seq_len + step], device=device, dtype=torch.long)
            out       = model(token.unsqueeze(0), cache_params=state,
                              cache_position=cache_pos, use_cache=True)
            state     = out.cache_params
            probs     = F.softmax(align(out.logits[:, -1, :]), dim=-1)
            token     = torch.argmax(probs, dim=-1)
            draft_ids.append(token.item())
            draft_probs.append(max(probs[0, token.item()].item(), 1e-9))

    return draft_ids, draft_probs

# ── Speculative decoding ──────────────────────────────────────
def speculative_generate(target_model, target_tok, draft_fn,
                         prompt, k=4, max_new_tokens=100):
    input_ids  = target_tok(prompt, return_tensors="pt").input_ids.to(DEVICE)
    output_ids = input_ids.clone()
    total_drafted = total_accepted = total_rollbacks = 0

    torch.cuda.synchronize()
    t0 = time.perf_counter()
    tokens_generated = 0

    while tokens_generated < max_new_tokens:
        draft_ids, draft_probs = draft_fn(output_ids, k)

        # ONE target forward pass
        draft_tensor = torch.tensor(draft_ids, device=DEVICE).unsqueeze(0)
        full_seq     = torch.cat([output_ids, draft_tensor], dim=1)
        with torch.no_grad():
            out = target_model(full_seq)

        prefix_len   = output_ids.shape[1]
        accepted_ids = []
        bonus_token  = None

        for j in range(k):
            pos          = prefix_len - 1 + j
            target_probs = F.softmax(out.logits[:, pos, :], dim=-1)
            t_vocab      = target_probs.shape[-1]
            did          = min(draft_ids[j], t_vocab - 1)  # clamp OOV
            p_target     = target_probs[0, did].item()
            p_draft      = draft_probs[j]
            accept_prob  = min(1.0, p_target / max(p_draft, 1e-9))
            accepted     = torch.rand(1).item() < accept_prob

            total_drafted += 1
            if accepted:
                total_accepted += 1
                accepted_ids.append(did)
            else:
                corrected      = target_probs[0].clone()
                corrected[did] = max(0.0, corrected[did].item() - p_draft)
                corrected      = torch.clamp(corrected, min=0.0)
                s              = corrected.sum()
                if s < 1e-6:
                    corrected = target_probs[0].clone()
                    s         = corrected.sum()
                corrected = corrected / s
                if torch.isnan(corrected).any() or torch.isinf(corrected).any():
                    corrected = torch.ones_like(corrected) / corrected.shape[0]
                bonus_token = torch.multinomial(corrected, 1).item()
                total_rollbacks += 1
                break

        if bonus_token is None:
            last_probs  = F.softmax(out.logits[:, prefix_len - 1 + k, :], dim=-1)
            bonus_token = torch.argmax(last_probs, dim=-1).item()

        new_ids    = accepted_ids + [bonus_token]
        output_ids = torch.cat([output_ids, torch.tensor(new_ids, device=DEVICE).unsqueeze(0)], dim=1)
        tokens_generated += len(new_ids)
        if bonus_token == target_tok.eos_token_id:
            break

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    alpha   = total_accepted / total_drafted if total_drafted > 0 else 0
    text    = target_tok.decode(output_ids[0][input_ids.shape[1]:], skip_special_tokens=True)
    return {
        "text": text, "tokens": tokens_generated,
        "time": elapsed, "tps": tokens_generated / max(elapsed, 1e-6),
        "acceptance_rate": alpha, "rollbacks": total_rollbacks,
    }

def autoregressive_baseline(model, tok, n_tokens, prompt):
    inputs = tok(prompt, return_tensors="pt").to(DEVICE)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=n_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    return n_tokens / elapsed

def run_n_times(fn, n):
    results = [fn() for _ in range(n)]
    return {k: sum(r[k] for r in results) / n
            for k in results[0] if isinstance(results[0][k], (int, float))}

# Test prompts
PROMPTS = {
    "factual":  "The first digits of pi are",
    "creative": "Once upon a time in a land far away",
    "code":     "def fibonacci(n):",
}
MAIN_PROMPT = PROMPTS["factual"]
N_TOKENS    = [50, 100, 200]
N_RUNS      = 3
K           = 4

print("All functions loaded ✓")

## Step 5 — Pair 2: TinyLlama-1.1B → Llama2-7B

**Tokenizer match: ✅ Same Llama tokenizer (32,000 vocab)**
This is fully valid speculative decoding — mathematically rigorous.

In [ ]:
# ── Load Pair 2 models ──────────────────────────────────────
print("Loading Pair 2 models...")
print("=" * 55)

# Draft: TinyLlama-1.1B — float16 (only 2.2 GB)
print("\nDraft model: TinyLlama-1.1B-Chat (float16)")
draft2_model, draft2_tok = load_model_float16(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    model_type="transformer"
)
vram_status("after TinyLlama")

# Target: Llama2-7B — int8 (~7 GB)
print("\nTarget model: Llama2-7B (int8 — ~7 GB)")
target2_model, target2_tok = load_model_int8(
    "meta-llama/Llama-2-7b-hf",
    model_type="transformer"
)
vram_status("after Llama2-7B")

In [ ]:
# ── Measure beta for Pair 2 ─────────────────────────────────
print("\nMeasuring β for Pair 2...")
beta2 = measure_beta(draft2_model, target2_model, target2_tok, MAIN_PROMPT)

In [ ]:
# ── Run Pair 2 benchmark ────────────────────────────────────
print("\nRunning Pair 2 benchmark...")
print("=" * 55)
print(f"{'n':>5} {'baseline':>10} {'spec_tps':>10} {'alpha':>7} {'speedup':>8}")
print("-" * 45)

pair2_results = {"beta": beta2, "pair_name": "TinyLlama-1.1B → Llama2-7B",
                 "same_vocab": True, "by_n": {}}

def tinyllama_draft(ids, k):
    return draft_transformer(draft2_model, ids, k)

for n in N_TOKENS:
    # Baseline
    bl_times = [autoregressive_baseline(target2_model, target2_tok, n, MAIN_PROMPT)
                for _ in range(N_RUNS)]
    baseline_tps = sum(bl_times) / N_RUNS

    # Speculative
    spec_res = [speculative_generate(target2_model, target2_tok,
                                     tinyllama_draft, MAIN_PROMPT,
                                     k=K, max_new_tokens=n)
                for _ in range(N_RUNS)]
    avg_tps   = sum(r["tps"]            for r in spec_res) / N_RUNS
    avg_alpha = sum(r["acceptance_rate"] for r in spec_res) / N_RUNS
    speedup   = avg_tps / baseline_tps

    pair2_results["by_n"][n] = {
        "baseline_tps": baseline_tps,
        "spec_tps":     avg_tps,
        "alpha":        avg_alpha,
        "speedup":      speedup,
        "output_text":  spec_res[0]["text"][:80],
    }
    print(f"{n:>5} {baseline_tps:>10.1f} {avg_tps:>10.1f} {avg_alpha:>7.2f} {speedup:>7.2f}×")

print("\nPair 2 complete ✓")

In [ ]:
# ── Correctness check for Pair 2 ────────────────────────────
print("Correctness check — Pair 2 (TinyLlama → Llama2-7B)")
print("Prompt: 'The first digits of pi are'")
r = speculative_generate(target2_model, target2_tok,
                          lambda ids, k: draft_transformer(draft2_model, ids, k),
                          "The first digits of pi are", k=4, max_new_tokens=50)
print(f"Output: {r['text'][:100]}")
print(f"α = {r['acceptance_rate']:.2f}  |  {r['tps']:.1f} tok/s")

In [ ]:
# ── Free VRAM before Pair 4 ─────────────────────────────────
print("Freeing VRAM before loading Pair 4...")
free_vram(draft2_model, target2_model)
del draft2_model, target2_model, draft2_tok, target2_tok
vram_status("after pair 2 cleanup")

## Step 6 — Pair 4: Mamba-130m → Llama2-7B

**Tokenizer match: ⚠️ Cross-architecture proxy (50,280 vs 32,000 vocab)**
Mamba uses GPT-NeoX tokenizer; Llama2 uses SentencePiece. Logits are aligned by min-vocab mapping.
This is a research experiment — not rigorous speculative decoding, but valid for architectural comparison.

In [ ]:
# ── Load Pair 4 models ──────────────────────────────────────
print("Loading Pair 4 models...")
print("=" * 55)

# Draft: Mamba-130m — float16 (only 0.26 GB)
print("\nDraft model: Mamba-130m (float16)")
draft4_model, draft4_tok = load_model_float16(
    "state-spaces/mamba-130m-hf",
    model_type="mamba"
)
vram_status("after Mamba-130m")

# Target: Llama2-7B — int8 (~7 GB)
# Note: use Llama2 tokenizer for target (and vocab alignment in the draft function)
print("\nTarget model: Llama2-7B (int8)")
target4_model, target4_tok = load_model_int8(
    "meta-llama/Llama-2-7b-hf",
    model_type="transformer"
)
vram_status("after Llama2-7B")

TARGET4_VOCAB = target4_model.config.vocab_size
print(f"\nMamba vocab : {draft4_model.config.vocab_size:,}")
print(f"Llama2 vocab: {TARGET4_VOCAB:,}")
print(f"Alignment   : truncate/pad Mamba logits to {TARGET4_VOCAB:,}")

In [ ]:
# ── Measure beta for Pair 4 ─────────────────────────────────
print("\nMeasuring β for Pair 4...")
# Use target tok for the input (Llama2 tokenizer)
beta4 = measure_beta(draft4_model, target4_model, target4_tok, MAIN_PROMPT)

In [ ]:
# ── Run Pair 4 benchmark ────────────────────────────────────
print("\nRunning Pair 4 benchmark...")
print("=" * 55)
print(f"{'n':>5} {'baseline':>10} {'spec_tps':>10} {'alpha':>7} {'speedup':>8}")
print("-" * 45)

pair4_results = {"beta": beta4, "pair_name": "Mamba-130m → Llama2-7B",
                 "same_vocab": False, "by_n": {}}

def mamba_llama_draft(ids, k):
    # Use Mamba draft but align logits to Llama2 vocab size
    return draft_mamba(draft4_model, ids, k, TARGET4_VOCAB)

for n in N_TOKENS:
    # Baseline
    bl_times = [autoregressive_baseline(target4_model, target4_tok, n, MAIN_PROMPT)
                for _ in range(N_RUNS)]
    baseline_tps = sum(bl_times) / N_RUNS

    # Speculative
    spec_res = [speculative_generate(target4_model, target4_tok,
                                     mamba_llama_draft, MAIN_PROMPT,
                                     k=K, max_new_tokens=n)
                for _ in range(N_RUNS)]
    avg_tps   = sum(r["tps"]            for r in spec_res) / N_RUNS
    avg_alpha = sum(r["acceptance_rate"] for r in spec_res) / N_RUNS
    speedup   = avg_tps / baseline_tps

    pair4_results["by_n"][n] = {
        "baseline_tps": baseline_tps,
        "spec_tps":     avg_tps,
        "alpha":        avg_alpha,
        "speedup":      speedup,
        "output_text":  spec_res[0]["text"][:80],
    }
    print(f"{n:>5} {baseline_tps:>10.1f} {avg_tps:>10.1f} {avg_alpha:>7.2f} {speedup:>7.2f}×")

print("\nPair 4 complete ✓")

In [ ]:
# ── Multi-prompt comparison for Pair 4 ─────────────────────
print("\nPair 4 — Multi-prompt acceptance rate comparison:")
for name, prompt in PROMPTS.items():
    r = speculative_generate(target4_model, target4_tok,
                              mamba_llama_draft, prompt, k=4, max_new_tokens=50)
    print(f"  {name:<10}: α={r['acceptance_rate']:.2f}  {r['tps']:.1f} tok/s")

In [ ]:
# ── Free VRAM ────────────────────────────────────────────────
print("\nFreeing VRAM after Pair 4...")
free_vram(draft4_model, target4_model)
del draft4_model, target4_model, draft4_tok, target4_tok
vram_status("after pair 4 cleanup")

## Step 7 — Save Results to JSON

In [ ]:
# Save both pair results
colab_results = {
    "pair2": pair2_results,
    "pair4": pair4_results,
    "metadata": {
        "gpu":      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "vram_gb":  torch.cuda.get_device_properties(0).total_memory/1e9 if torch.cuda.is_available() else 0,
        "k":        K,
        "n_runs":   N_RUNS,
        "prompt":   MAIN_PROMPT,
    }
}

with open("colab_results_pairs_2_4.json", "w") as f:
    json.dump(colab_results, f, indent=2)

print("Results saved to colab_results_pairs_2_4.json")
print("\nQuick summary:")
for pk, res in [("pair2", pair2_results), ("pair4", pair4_results)]:
    tps100 = res["by_n"][100]["spec_tps"]
    su100  = res["by_n"][100]["speedup"]
    al100  = res["by_n"][100]["alpha"]
    print(f"  {res['pair_name']:<35} {tps100:.1f} tok/s  α={al100:.2f}  {su100:.2f}×")

## Step 8 — Generate Comparison Charts

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.facecolor": "#0F0225", "axes.facecolor": "#0F0225",
    "axes.edgecolor": "#B0A0CC", "axes.labelcolor": "#B0A0CC",
    "xtick.color": "#B0A0CC", "ytick.color": "#B0A0CC",
    "text.color": "#FFFFFF", "grid.color": "#2E0D55", "grid.alpha": 0.5,
    "legend.facecolor": "#1A0533", "legend.edgecolor": "#B0A0CC",
    "font.family": "DejaVu Sans",
})

COLORS = {"pair2": "#F5C842", "pair4": "#FF6B6B"}
LABELS = {"pair2": "TinyLlama-1.1B → Llama2-7B",
          "pair4": "Mamba-130m → Llama2-7B"}
N_LIST = [50, 100, 200]
results = {"pair2": pair2_results, "pair4": pair4_results}

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Pairs 2 & 4 — Speculative Decoding on Google Colab T4",
             fontsize=14, color="white", y=1.01)

# ── Chart 1: Throughput ───────────────────────────────────────
ax = axes[0, 0]
ax.set_title("Token Throughput (tok/s)", fontsize=12)
ax.set_xlabel("Sequence length (n)")
ax.set_ylabel("Tokens / second")
ax.grid(alpha=0.4)
ax.set_axisbelow(True)
for pk in ["pair2", "pair4"]:
    r   = results[pk]
    tps = [r["by_n"][n]["spec_tps"] for n in N_LIST]
    bl  = [r["by_n"][n]["baseline_tps"] for n in N_LIST]
    ax.plot(N_LIST, tps, "o-", color=COLORS[pk], linewidth=2.5,
            markersize=8, label=LABELS[pk] + " (spec)")
    ax.plot(N_LIST, bl,  "s--", color=COLORS[pk], linewidth=1.5,
            markersize=6, alpha=0.6, label=LABELS[pk] + " (target only)")
ax.legend(fontsize=8)

# ── Chart 2: Speedup ─────────────────────────────────────────
ax = axes[0, 1]
ax.set_title("Speedup over Baseline (×)", fontsize=12)
ax.set_xlabel("Sequence length (n)")
ax.set_ylabel("Speedup (×)")
ax.axhline(1.0, color="#FF6B6B", linestyle="--", alpha=0.5)
ax.grid(alpha=0.4)
ax.set_axisbelow(True)
bw = 0.35
x  = np.arange(len(N_LIST))
for i, pk in enumerate(["pair2", "pair4"]):
    speedups = [results[pk]["by_n"][n]["speedup"] for n in N_LIST]
    off      = (i - 0.5) * bw
    bars     = ax.bar(x + off, speedups, bw, color=COLORS[pk],
                      label=LABELS[pk], edgecolor="#0F0225", linewidth=0.5)
    for bar, v in zip(bars, speedups):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{v:.2f}×", ha="center", va="bottom", fontsize=9, color="white")
ax.set_xticks(x)
ax.set_xticklabels([f"n={n}" for n in N_LIST])
ax.legend(fontsize=8)

# ── Chart 3: Acceptance rate ──────────────────────────────────
ax = axes[1, 0]
ax.set_title("Acceptance Rate α per Sequence Length", fontsize=12)
ax.set_xlabel("Sequence length (n)")
ax.set_ylabel("Acceptance rate α")
ax.set_ylim(0, 1.0)
ax.grid(alpha=0.4)
ax.set_axisbelow(True)
for pk in ["pair2", "pair4"]:
    alphas = [results[pk]["by_n"][n]["alpha"] for n in N_LIST]
    ax.plot(N_LIST, alphas, "o-", color=COLORS[pk], linewidth=2.5,
            markersize=8, label=LABELS[pk])
ax.legend(fontsize=8)
# Annotation: same vocab vs proxy
ax.text(0.05, 0.95, "Pair 2: same Llama tokenizer ✓",
        transform=ax.transAxes, fontsize=8.5,
        color=COLORS["pair2"], va="top")
ax.text(0.05, 0.88, "Pair 4: cross-arch proxy ⚠",
        transform=ax.transAxes, fontsize=8.5,
        color=COLORS["pair4"], va="top")

# ── Chart 4: Beta + theoretical speedup ──────────────────────
ax = axes[1, 1]
ax.set_title("Cost Ratio β and Theoretical Speedup", fontsize=12)

labels = [LABELS[pk].split("→")[0].strip() for pk in ["pair2", "pair4"]]
betas  = [results[pk]["beta"] for pk in ["pair2", "pair4"]]
colors = [COLORS[pk] for pk in ["pair2", "pair4"]]

# Calculate theoretical speedup per pair
th_speedups = []
for pk in ["pair2", "pair4"]:
    b     = results[pk]["beta"]
    alpha = results[pk]["by_n"][100]["alpha"]
    k     = K
    if 0 < alpha < 1 and b > 0:
        num = (1 - alpha**(k+1)) / (1 - alpha)
        den = k * b + 1
        th_speedups.append(num / den)
    else:
        th_speedups.append(0)

x2  = np.array([0, 1])
bw2 = 0.3
b1  = ax.bar(x2 - bw2/2, betas, bw2, color=colors, edgecolor="#0F0225",
             label="Cost ratio β", linewidth=0.5)
ax2r = ax.twinx()
ax2r.set_ylabel("Theoretical speedup (×)", color="#F5C842")
b2   = ax2r.bar(x2 + bw2/2, th_speedups, bw2, color=colors, alpha=0.5,
                edgecolor="#0F0225", hatch="//", linewidth=0.5)

for bar, v in zip(b1, betas):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"β={v:.3f}", ha="center", va="bottom", fontsize=9, color="white")
for bar, v in zip(b2, th_speedups):
    ax2r.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
              f"{v:.2f}×", ha="center", va="bottom", fontsize=9, color=COLORS["pair2"])

ax.set_xticks(x2)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("β (lower = cheaper draft)")
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='gray', label='β (solid)'),
                   Patch(facecolor='gray', hatch='//', alpha=0.5, label='Theoretical speedup (hatched)')]
ax.legend(handles=legend_elements, fontsize=8)

plt.tight_layout()
plt.savefig("colab_pairs_2_4_results.png", dpi=150, bbox_inches="tight",
            facecolor="#0F0225")
plt.show()
print("Chart saved: colab_pairs_2_4_results.png")

## Step 9 — Final Summary

In [ ]:
print("=" * 65)
print("  FINAL RESULTS SUMMARY — Pairs 2 & 4 on Colab")
print("=" * 65)
print(f"  GPU      : {colab_results['metadata']['gpu']}")
print(f"  VRAM     : {colab_results['metadata']['vram_gb']:.1f} GB")
print(f"  k        : {K}  |  Runs per config: {N_RUNS}")
print()
print(f"  {'Metric':<30} {'Pair 2':>12} {'Pair 4':>12}")
print(f"  {'─'*56}")

metrics = [
    ("β (cost ratio)",            "beta",       None),
    ("Baseline tok/s (n=100)",    None,         ("baseline_tps", 100)),
    ("Speculative tok/s (n=50)",  None,         ("spec_tps", 50)),
    ("Speculative tok/s (n=100)", None,         ("spec_tps", 100)),
    ("Speculative tok/s (n=200)", None,         ("spec_tps", 200)),
    ("Acceptance rate α (n=100)", None,         ("alpha", 100)),
    ("Speedup × (n=50)",          None,         ("speedup", 50)),
    ("Speedup × (n=100)",         None,         ("speedup", 100)),
    ("Speedup × (n=200)",         None,         ("speedup", 200)),
]

for label, top_key, by_n_key in metrics:
    if top_key:
        v2 = pair2_results.get(top_key, 0)
        v4 = pair4_results.get(top_key, 0)
    else:
        key, n = by_n_key
        v2 = pair2_results["by_n"][n].get(key, 0)
        v4 = pair4_results["by_n"][n].get(key, 0)
    print(f"  {label:<30} {v2:>12.3f} {v4:>12.3f}")

print()
print("  Vocab alignment:")
print(f"    Pair 2: {'✓ Same Llama tokenizer (32,000 vocab)'}")
print(f"    Pair 4: {'⚠ Cross-arch proxy (50,280 → 32,000 aligned)'}")
print()
print("  Output text sample (pair 2, pi prompt):")
print(f"    {pair2_results['by_n'][100]['output_text']}")
print("  Output text sample (pair 4, pi prompt):")
print(f"    {pair4_results['by_n'][100]['output_text']}")
print("=" * 65)

## Step 10 — Download Results

In [ ]:
# Download files to your computer
from google.colab import files

print("Downloading result files...")
files.download("colab_results_pairs_2_4.json")
files.download("colab_pairs_2_4_results.png")
print("Downloaded: colab_results_pairs_2_4.json")
print("Downloaded: colab_pairs_2_4_results.png")
print()
print("Copy these files into your speculative_decoding/ folder")
print("Then run: python comparison_plots.py to include in the 4-way comparison")

## Step 11 (Optional) — Acceptance Rate vs k Sweep

This sweeps k from 2 to 8 for both pairs at n=100 — gives you the α sensitivity chart.

In [ ]:
# This cell reloads both models and sweeps k values
# Only run if you have time — takes ~15 min

RUN_K_SWEEP = False  # Set to True to run

if RUN_K_SWEEP:
    # Reload Pair 2
    print("Reloading Pair 2 for k sweep...")
    draft2_ks, _ = load_model_float16("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
    target2_ks, tok2_ks = load_model_int8("meta-llama/Llama-2-7b-hf")

    k_sweep_p2 = {}
    for k_val in [2, 3, 4, 5, 6, 8]:
        r = speculative_generate(target2_ks, tok2_ks,
                                  lambda ids, k: draft_transformer(draft2_ks, ids, k),
                                  MAIN_PROMPT, k=k_val, max_new_tokens=100)
        k_sweep_p2[k_val] = {"tps": r["tps"], "alpha": r["acceptance_rate"]}
        print(f"  Pair 2 k={k_val}: α={r['acceptance_rate']:.2f}  {r['tps']:.1f} tok/s")

    free_vram(draft2_ks, target2_ks)
    del draft2_ks, target2_ks, tok2_ks

    # Add to results
    colab_results["k_sweep_pair2"] = k_sweep_p2
    with open("colab_results_pairs_2_4.json", "w") as f:
        json.dump(colab_results, f, indent=2)
    print("k sweep saved.")
else:
    print("Set RUN_K_SWEEP = True to run this cell.")